# 01 · Spend analysis

Where the money goes in the synthetic purchasing base: spend by family and month, ABC Pareto, single-source exposure and purchase price variance (PPV) against the last price paid.

Build the base first with `python -m src.generate.purchasing` and check it with `python -m src.validate`. All numbers come from the views in `sql/views.sql`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.db import DEFAULT_DB, connect, create_schema

con = connect(DEFAULT_DB)
create_schema(con)  # refreshes the views


def q(sql):
    return con.execute(sql).df()


musd = mtick.FuncFormatter(lambda v, _: f"{v / 1e6:,.0f}M")

## Spend by family

In [ ]:
family = q("""
    SELECT family, spend_usd / 2 AS spend_per_year
    FROM v_spend_cube
    WHERE family IS NOT NULL AND country IS NULL AND year IS NULL
    ORDER BY spend_per_year
""")
ax = family.plot.barh(x="family", y="spend_per_year", legend=False, figsize=(8, 4))
ax.xaxis.set_major_formatter(musd)
ax.set(xlabel="USD per year", ylabel="", title="Annual spend by part family")
family.assign(
    usd_m_per_year=(family.spend_per_year / 1e6).round(2),
    share=(family.spend_per_year / family.spend_per_year.sum()).round(3),
).drop(columns="spend_per_year")

## Monthly spend

In [ ]:
monthly = q("""
    SELECT month, sum(spend_usd) AS spend_usd
    FROM v_po_lines GROUP BY month ORDER BY month
""")
ax = monthly.plot(x="month", y="spend_usd", legend=False, figsize=(8, 3), marker="o")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v / 1e6:,.1f}M"))
ax.set(xlabel="", ylabel="USD", title="Monthly spend");

## ABC Pareto

An item is class A while the spend before it is under 80% of the total, B under 95%, C otherwise.

In [ ]:
abc = q("SELECT * FROM v_abc ORDER BY rank")
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(abc["rank"] / len(abc), abc["cum_share"])
for cut in (0.80, 0.95):
    ax.axhline(cut, ls="--", lw=0.8, color="grey")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set(xlabel="Share of materials", ylabel="Cumulative share of spend", title="Pareto curve")
abc.groupby("abc_class").agg(
    materials=("material_id", "count"), spend_share=("spend_share", "sum")
).round(3)

## Single-source exposure

In [ ]:
q("SELECT * FROM v_single_source").round(3)

## Purchase price variance

PPV compares each order with the last price paid for the same material, from any supplier. Positive means the buyer paid more than last time.

In [ ]:
ppv = q("SELECT * FROM v_ppv WHERE ppv_usd IS NOT NULL")
by_month = ppv.groupby("month", as_index=False)["ppv_usd"].sum()
fig, ax = plt.subplots(figsize=(9, 3))
colors = ["tab:red" if v > 0 else "tab:green" for v in by_month["ppv_usd"]]
ax.bar(range(len(by_month)), by_month["ppv_usd"], color=colors)
ax.set_xticks(range(len(by_month)), [m.strftime("%Y-%m") for m in by_month["month"]], rotation=90)
ax.axhline(0, color="black", lw=0.8)
ax.set(ylabel="USD", title="Monthly PPV against last price paid (red = paid more)")
print(f"Total PPV: USD {ppv['ppv_usd'].sum():,.0f}")

In [ ]:
top = (
    ppv.groupby(["material_id", "family"], as_index=False)["ppv_usd"]
    .sum()
    .assign(abs_ppv=lambda d: d["ppv_usd"].abs())
    .nlargest(10, "abs_ppv")
    .drop(columns="abs_ppv")
)
top.round(0)

## Reading

With synthetic copper, PPV mostly tracks the copper path three months earlier plus order noise and supplier switches on multi-source items. Large PPV on a multi-source material usually means volume moved between suppliers with different price levels, which is the first hint of the overpriced suppliers the should-cost model will test.